# Adam: adaptive moment estimation, собранный руками

Статья: Kingma & Ba, *Adam: A Method for Stochastic Optimization*, ICLR 2015 (arXiv:1412.6980).
Узел дерева «100 дней»: `dl_adam`.

Что строим и обучаем: одно-слойный MLP `64 -> 32 -> 10` на `sklearn.load_digits` (реальный MNIST-подобный таск, 1797 картинок 8x8, 10 классов). Forward уже дан; ты пишешь backprop, **сам оптимизатор Adam** (вклад статьи), его абляцию до чистого SGD и тренировочный цикл. Всё на numpy — алгоритм статьи видно построчно, не спрятан за `torch.optim`.

Компьют: CPU, обучение за секунды. GPU не нужен.

Parts:
1. Baseline-пол — тривиальный предсказатель, планку которого обязан побить любой оптимизатор.
2. Backprop через MLP — градиенты руками (ядро любого обучаемого метода).
3. Тренировочный цикл — минибатчи, история лосса.
4. **Adam.step** — вклад статьи: моментные EMA + bias correction + адаптивный шаг.
5. SGD.step — абляция: та же петля с выключенной моментной машинерией.
6. Headline-замер: Adam vs SGD на равных шагах и lr.
7. Фишка: bias correction и что без неё ломается на первых шагах.

Метрика задачи: train cross-entropy на равном числе шагов и одном базовом lr (контраст «с моментами vs без»), плюс test accuracy и «сколько эпох до целевого лосса».

## Эксперименты-первыми

До метода — три слота, заполни от руки после того, как прочитаешь §6 статьи (или прикинешь из того, что знаешь про Adam).

1. **Что показали vs что обещал abstract.** Abstract обещает метод, который «computationally efficient, little memory, invariant to diagonal rescaling of gradients». Что из §6.2 (MNIST MLP) реально видно на графиках — ...

2. **Самый слабый эксперимент статьи.** Где сравнение выглядит менее убедительно (выбор lr для бейзлайнов? одна архитектура? train loss, не test?) — ...

3. **Где метод ломается / оговорки авторов.** Что сами Kingma & Ba отмечают как хрупкое (например, поведение $\hat v$ на разреженных градиентах, роль $\epsilon$, поздняя нестабильность) — ...

Заметки останутся в ноутбуке как твой конспект. Никто не проверяет «правильность» — это разметка внимания перед кодом.

## Зачем это

SGD с фиксированным lr неявно предполагает, что все координаты градиента одного масштаба. В нейросети это неправда: градиенты по разным слоям и разным весам гуляют на порядки, и один $\alpha$ либо расходится на «крутых» координатах, либо ползёт на «пологих». Классический ответ — подбирать lr вручную и по слоям; дорого и хрупко.

Фишка Adam: держать для каждого параметра две скользящие оценки — среднее градиента ($m$, «куда обычно тянет») и среднее квадрата ($v$, «насколько шумно/круто здесь»), и делать шаг $\propto m/\sqrt{v}$. Координата с большими градиентами получает большой $\sqrt v$ в знаменателе и автоматически укорачивает шаг; тихая координата — удлиняет. Это и есть инвариантность к диагональному масштабированию: адаптивный per-coordinate lr без ручной настройки по слоям.

Преимущество над альтернативой (RMSProp/AdaGrad): AdaGrad копит сумму квадратов и монотонно душит lr до нуля; RMSProp чинит это через EMA, но не корректирует смещение старта. Adam добавляет момент ($m$) И bias correction — и именно это даёт быстрый, устойчивый старт, который мы увидим в Part 7.

Данные — `load_digits`, потому что это тот же жанр, что MNIST-MLP из §6.2 статьи, но обучается за секунды на CPU: контраст оптимизаторов виден без часа ожидания.

## Теория (по Фейнману)

**Задача.** Минимизируем $L(\theta)$ по стохастическим градиентам $g_t = \nabla_\theta L_t(\theta_{t-1})$.

**Механизм.** Adam (Algorithm 1) на шаге $t$ (1-based) держит два экспоненциальных средних:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\, g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\, g_t^2$$

Оба стартуют с нуля, поэтому в начале они смещены к нулю. **Bias correction** убирает это смещение:

$$\hat m_t = \frac{m_t}{1-\beta_1^{\,t}}, \qquad \hat v_t = \frac{v_t}{1-\beta_2^{\,t}}$$

Шаг:

$$\theta_t = \theta_{t-1} - \alpha\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}$$

**Что метод измеряет.** $m$ — оценка первого момента (направление), $v$ — второго (масштаб/дисперсия). Отношение $\hat m/\sqrt{\hat v}$ — сигнал-к-масштабу по каждой координате; статья показывает, что при $\epsilon\to 0$ эффективный шаг ограничен $|\Delta_t| \lesssim \alpha$ (доверительный интервал шага), — почему Adam устойчив.

**Почему bias correction важна именно рано.** На $t=1$: $m_1=(1-\beta_1)g$, $v_1=(1-\beta_2)g^2$. Без коррекции $m_1/\sqrt{v_1} = \frac{(1-\beta_1)}{\sqrt{1-\beta_2}}\,\frac{g}{|g|} \approx 3.16\,\mathrm{sign}(g)$ — шаг раздут в ~3.16 раза. С коррекцией делители сокращаются: $\hat m_1/\sqrt{\hat v_1} = g/|g| = \mathrm{sign}(g)$, шаг ровно $\approx\alpha$. Это ты померяешь в Part 7.

**Где у тебя дыры.** `backward` (градиенты), `train` (цикл), `Adam.step` (вклад статьи), `SGD.step` (абляция), `trivial_floor` (пол).

❓ **Inline Q0.** AdaGrad делит на $\sqrt{\sum_{\tau\le t} g_\tau^2}$, Adam — на $\sqrt{\hat v_t}$ с EMA. Почему при долгом обучении AdaGrad почти останавливается, а Adam — нет? Что именно меняет замена суммы на скользящее среднее?

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DEVICE = "cpu"   # чистый numpy; фиксируем для паритета со спекой задачи
SEED = 0


def load_data(seed=SEED):
    ds = load_digits()
    X, y = ds.data.astype(np.float64), ds.target.astype(np.int64)
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.25, random_state=seed, stratify=y
    )
    scaler = StandardScaler().fit(Xtr)
    Xtr = scaler.transform(Xtr)
    Xte = scaler.transform(Xte)
    return Xtr, ytr, Xte, yte


def init_params(n_in, n_hid, n_out, seed=SEED):
    rng = np.random.default_rng(seed)
    # He init для ReLU-слоя, поменьше (Glorot-ish) для выхода.
    return {
        "W1": rng.standard_normal((n_in, n_hid)) * np.sqrt(2.0 / n_in),
        "b1": np.zeros(n_hid),
        "W2": rng.standard_normal((n_hid, n_out)) * np.sqrt(1.0 / n_hid),
        "b2": np.zeros(n_out),
    }


def _softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def forward(params, X):
    """Дан готовым. Возвращает (p, cache), где p — softmax-вероятности [B,10],
    cache = (X, z1, a1, p) — всё, что понадобится backward."""
    z1 = X @ params["W1"] + params["b1"]
    a1 = np.maximum(z1, 0.0)          # ReLU
    z2 = a1 @ params["W2"] + params["b2"]
    p = _softmax(z2)
    cache = (X, z1, a1, p)
    return p, cache


def cross_entropy(p, y):
    n = y.shape[0]
    return -np.mean(np.log(p[np.arange(n), y] + 1e-12))


def accuracy(params, X, y):
    p, _ = forward(params, X)
    return float((p.argmax(axis=1) == y).mean())


Xtr, ytr, Xte, yte = load_data()
print(f"device={DEVICE}  train={Xtr.shape}  test={Xte.shape}  classes={int(ytr.max())+1}")
LR, EPOCHS, BATCH = 1e-3, 30, 64   # lr = paper default alpha, один и тот же для ОБОИХ оптимизаторов (честная абляция)

## Part 1 — baseline-пол

Прежде чем радоваться, что оптимизатор «работает», нужна планка. Тривиальный предсказатель, который ничего не учит: угадывает по априорному распределению классов. Его train cross-entropy и test accuracy — пол, ниже которого падать стыдно; любой настоящий оптимизатор обязан его пробить.

In [ ]:
def trivial_floor(ytr, yte):
    """Планка необучаемого предсказателя.

    Args:
        ytr: [Ntr] int — метки трейна (классы 0..K-1).
        yte: [Nte] int — метки теста.
    Returns:
        (ce, acc): ce float — train cross-entropy предсказателя, который на
            каждом примере выдаёт эмпирическое априорное распределение классов
            трейна; acc float — доля теста, угаданная константным предсказанием
            самого частого класса трейна.
    Note: используй тот же лог-клиппинг 1e-12, что и cross_entropy выше.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
ce, acc = trivial_floor(ytr, yte)
# Expected: ce ~ log(10) ~ 2.3025 (равномернее -> ближе к log K), acc ~ 0.10
assert np.isfinite(ce) and np.isfinite(acc), "пол должен быть конечным"
assert 2.20 < ce < 2.31, f"CE пола ~ log(10)=2.3025, получил {ce:.4f} — не то распределение?"
assert 0.05 < acc < 0.20, f"majority-accuracy ~0.10 на 10 сбалансированных классах, получил {acc:.4f}"
print(f"ok  floor: train_CE={ce:.4f}  test_acc={acc:.4f}")

## Part 2 — backprop через MLP

Оптимизатор без градиентов — пустой цикл. `forward` дан; напиши обратный проход. Сеть: $z_1 = XW_1+b_1$, $a_1=\mathrm{ReLU}(z_1)$, $z_2=a_1W_2+b_2$, $p=\mathrm{softmax}(z_2)$, лосс — средняя cross-entropy по батчу. Нужны градиенты по всем четырём параметрам, усреднённые по батчу (та же нормировка, что в лоссе).

❓ **Inline Q1.** Для softmax+CE градиент по логитам $z_2$ имеет знаменитую замкнутую форму. Какую, и почему она не содержит явного деления на вероятности (в отличие от «наивного» CE по отдельности)?

In [ ]:
def backward(params, cache, y):
    """Градиенты средней cross-entropy по параметрам MLP.

    Args:
        params: dict с "W1","b1","W2","b2" — те же веса, что в forward.
        cache: (X, z1, a1, p) — ровно то, что вернул forward.
        y: [B] int — истинные классы батча.
    Returns:
        grads: dict с ключами "W1","b1","W2","b2"; каждый градиент той же
            формы, что соответствующий параметр, и усреднён по батчу (та же
            нормировка 1/B, что в лоссе), чтобы масштаб не зависел от размера батча.
    Note: ReLU не пропускает градиент там, где вход был <= 0.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# grad-check: аналитический backward против численного градиента на маленьком батче.
rng = np.random.default_rng(1)
p0 = init_params(64, 32, 10, seed=1)
Xb, yb = Xtr[:16], ytr[:16]
_, cache = forward(p0, Xb)
grads = backward(p0, cache, yb)

for k in ("W1", "b1", "W2", "b2"):
    assert grads[k].shape == p0[k].shape, f"{k}: форма градиента {grads[k].shape} != {p0[k].shape}"

def loss_at(P):
    pp, _ = forward(P, Xb)
    return cross_entropy(pp, yb)

# численная проверка на нескольких случайных координатах каждого параметра
eps = 1e-5
max_rel = 0.0
for k in ("W1", "b1", "W2", "b2"):
    flat = p0[k].reshape(-1)
    for idx in rng.choice(flat.size, size=min(5, flat.size), replace=False):
        orig = flat[idx]
        flat[idx] = orig + eps; lp = loss_at(p0)
        flat[idx] = orig - eps; lm = loss_at(p0)
        flat[idx] = orig
        num = (lp - lm) / (2 * eps)
        ana = grads[k].reshape(-1)[idx]
        rel = abs(num - ana) / (abs(num) + abs(ana) + 1e-12)
        max_rel = max(max_rel, rel)

# Expected: max relative error << 1e-4 при аналитически верном backward
assert max_rel < 1e-4, f"grad-check провален, max rel err={max_rel:.2e} — проверь ReLU'-маску и нормировку 1/B"
print(f"ok  backward: max rel err vs numeric = {max_rel:.2e}")

## Part 3 — тренировочный цикл

Одна петля, общая для всех оптимизаторов (в этом весь смысл честной абляции: меняем только `opt.step`). Минибатчи случайной перестановкой, после каждой эпохи — полный train cross-entropy в историю. Оптимизатор передаётся как объект с методом `.step(params, grads)`, который обновляет `params` на месте.

Классы `Adam` и `SGD` появятся в Part 4-5; здесь ты пишешь только цикл, а `make_opt` соберёт нужный объект (дан ниже).

In [ ]:
def train(make_opt, Xtr, ytr, epochs=30, batch=64, seed=SEED):
    """Обучение MLP минибатчами; оптимизатор абстрагирован.

    Args:
        make_opt: callable(params) -> optimizer, где у optimizer есть метод
            step(params, grads), обновляющий params на месте.
        Xtr: [N, d] float — признаки трейна.
        ytr: [N] int — метки трейна.
        epochs: int — сколько полных проходов по данным.
        batch: int — размер минибатча.
        seed: int — для инициализации весов и перемешивания (воспроизводимость).
    Returns:
        (params, history): params — обученные веса (dict); history — список
            длины epochs, где элемент t — полный train cross-entropy ПОСЛЕ
            эпохи t (на всём Xtr, не на последнем батче).
    Note: перемешивай индексы каждую эпоху из ОДНОГО rng(seed), чтобы прогоны
        двух оптимизаторов видели идентичные батчи.
    """
    params = init_params(Xtr.shape[1], 32, 10, seed=seed)
    opt = make_opt(params)
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# Санити цикла на игрушечном полном-батч GD (без Adam/SGD-классов, чтобы изолировать петлю).
class _FullGD:
    def __init__(self, params, lr=0.05):
        self.lr = lr
    def step(self, params, grads):
        for k in params:
            params[k] -= self.lr * grads[k]

params, hist = train(lambda P: _FullGD(P, lr=0.05), Xtr, ytr, epochs=8, batch=64, seed=0)
assert len(hist) == 8, f"history должна быть длины epochs=8, получил {len(hist)}"
assert all(np.isfinite(hist)), "в history есть nan/inf"
# лосс обязан монотонно-ish падать при разумном lr: конец заметно ниже старта
assert hist[-1] < hist[0] - 0.1, f"лосс не падает: старт {hist[0]:.3f} -> конец {hist[-1]:.3f}; цикл обновляет веса?"
# детерминизм по seed
_, hist2 = train(lambda P: _FullGD(P, lr=0.05), Xtr, ytr, epochs=8, batch=64, seed=0)
assert np.allclose(hist, hist2), "один seed -> разные истории: перемешивание не детерминировано"
print(f"ok  train loop: CE {hist[0]:.3f} -> {hist[-1]:.3f}, детерминизм по seed")

## Part 4 — Adam.step (вклад статьи)

Сердце ассайнмента. Реализуй один шаг Adam из Algorithm 1. Класс уже держит состояние: счётчик шагов `self.t`, словари первых и вторых моментов `self.m`, `self.v` (оба инициализированы нулями по форме параметров), гиперпараметры `self.lr, self.b1, self.b2, self.eps` и флаг `self.bias_correction`.

Математика — в блоке «Теория» выше; здесь тебе нужно превратить её в обновление `params` на месте. Флаг `bias_correction=False` — это второй контроль для Part 7: он должен пропускать шаг коррекции смещения, всё остальное без изменений.

❓ **Inline Q2.** Почему $\epsilon$ стоит именно в знаменателе как $\sqrt{\hat v}+\epsilon$, а не, скажем, $\sqrt{\hat v + \epsilon}$? Что произойдёт с шагом по координате, где градиент долго был нулевым, при каждом из вариантов?

In [ ]:
class Adam:
    """Adaptive Moment Estimation (Kingma & Ba 2014, Algorithm 1).

    Состояние (готово): self.t (счётчик шагов, стартует с 0), self.m/self.v
    (dict моментов по форме params, нули), self.lr/self.b1/self.b2/self.eps,
    self.bias_correction (bool).
    """

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8,
                 bias_correction=True):
        self.lr = lr
        self.b1 = beta1
        self.b2 = beta2
        self.eps = eps
        self.bias_correction = bias_correction
        self.t = 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}

    def step(self, params, grads):
        """Один шаг Adam; обновляет каждый params[k] на месте.

        Args:
            params: dict весов; изменить in-place.
            grads: dict градиентов, те же ключи и формы, что params.
        Returns:
            None. Побочный эффект: params сдвинуты на адаптивный per-coordinate
            шаг, self.t увеличен, self.m/self.v обновлены. При
            self.bias_correction=False шаг делается по несмещённым (сырым)
            оценкам моментов.
        Note: t — 1-based на момент вычисления коррекции (увеличь ДО шага).
        """
        # YOUR CODE
        raise NotImplementedError

In [ ]:
# 1) Единичный шаг на константном градиенте: при bias correction эффективный шаг ~ lr.
p = {"w": np.array([0.0, 0.0])}
g = {"w": np.array([2.0, -5.0])}   # разный масштаб координат
opt = Adam(p, lr=1e-3)
opt.step(p, g)
# t=1, bias-corrected: dw = -lr * sign(g) = -/+1e-3 независимо от |g| (инвариантность к масштабу)
assert np.allclose(p["w"], np.array([-1e-3, +1e-3]), atol=1e-9), \
    f"первый шаг Adam должен быть -lr*sign(g) (инвариант к масштабу), получил {p['w']}"
assert opt.t == 1, f"self.t должен быть 1 после одного шага, а не {opt.t}"

# 2) Без bias correction тот же первый шаг РАЗДУТ множителем (1-b1)/sqrt(1-b2) ~ 3.16.
p2 = {"w": np.array([0.0])}
opt2 = Adam(p2, lr=1e-3, bias_correction=False)
opt2.step(p2, {"w": np.array([7.0])})
factor = abs(p2["w"][0]) / 1e-3
assert 3.0 < factor < 3.3, f"несмещённый первый шаг ~3.16*lr, получил фактор {factor:.3f}"

# 3) Адаптивность на нескольких шагах: координата с крупным градиентом не улетает.
p3 = {"w": np.array([0.0, 0.0])}
optc = Adam(p3, lr=1e-2)
for _ in range(50):
    optc.step(p3, {"w": np.array([100.0, 0.01])})
assert np.all(np.isfinite(p3["w"])), "Adam разошёлся (nan/inf)"
assert abs(p3["w"][0]) < 60 * 1e-2, "крупноградиентная координата не должна улетать — где sqrt(v) в знаменателе?"
print("ok  Adam.step: масштаб-инвариантный первый шаг, эффект bias correction, устойчивость")

## Part 5 — SGD.step (абляция)

Абляция «выключить фишку»: тот же интерфейс `.step(params, grads)`, но без моментов и без адаптивного знаменателя — чистый градиентный спуск $\theta \leftarrow \theta - \alpha g$. Это «Adam с выключенной моментной машинерией», наш контроль для headline-сравнения.

In [ ]:
class SGD:
    """Абляция Adam: моментная машинерия убрана. Ванильный градиентный спуск."""

    def __init__(self, params, lr=1e-3):
        self.lr = lr

    def step(self, params, grads):
        """Один шаг ванильного SGD; обновляет params на месте.

        Args:
            params: dict весов; изменить in-place.
            grads: dict градиентов, те же ключи/формы.
        Returns:
            None. Никакого адаптивного масштабирования, никакого состояния.
        """
        # YOUR CODE
        raise NotImplementedError

In [ ]:
p = {"w": np.array([1.0, -2.0])}
SGD(p, lr=0.1).step(p, {"w": np.array([0.5, 4.0])})
assert np.allclose(p["w"], np.array([1.0 - 0.1*0.5, -2.0 - 0.1*4.0])), \
    f"SGD-шаг = theta - lr*g, получил {p['w']}"
print("ok  SGD.step")

## Part 6 — headline: Adam vs SGD на равных условиях

Всё вместе. Один lr, одно число шагов, одна петля, одни батчи — различается только `.step`. Именно так статья показывает эффект: контраст «с моментной машинерией (Adam) vs без (SGD)». Смотрим финальный train CE, test accuracy и скорость сходимости (сколько эпох каждому нужно, чтобы догнать лосс, который у Adam есть к эпохе 5).

In [ ]:
p_adam, h_adam = train(lambda P: Adam(P, lr=LR), Xtr, ytr, epochs=EPOCHS, batch=BATCH, seed=SEED)
p_sgd,  h_sgd  = train(lambda P: SGD(P, lr=LR),  Xtr, ytr, epochs=EPOCHS, batch=BATCH, seed=SEED)

floor_ce, floor_acc = trivial_floor(ytr, yte)
adam_final, sgd_final = h_adam[-1], h_sgd[-1]
adam_acc, sgd_acc = accuracy(p_adam, Xte, yte), accuracy(p_sgd, Xte, yte)

target = h_adam[4]   # лосс Adam к эпохе 5 — прокси скорости
def epochs_to(hist, tgt):
    for i, v in enumerate(hist):
        if v <= tgt:
            return i + 1
    return None
adam_speed, sgd_speed = epochs_to(h_adam, target), epochs_to(h_sgd, target)

print("===== metrics =====")
print(f"trivial floor : train_CE={floor_ce:.4f}  test_acc={floor_acc:.4f}")
print(f"SGD  (lr={LR}) : train_CE={sgd_final:.4f}  test_acc={sgd_acc:.4f}")
print(f"Adam (lr={LR}) : train_CE={adam_final:.4f}  test_acc={adam_acc:.4f}")
print(f"epochs to reach {target:.3f}: Adam={adam_speed}  SGD={sgd_speed}")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, EPOCHS + 1), h_adam, label="Adam")
    plt.plot(range(1, EPOCHS + 1), h_sgd, label="SGD")
    plt.axhline(floor_ce, ls="--", c="gray", label="trivial floor")
    plt.xlabel("epoch"); plt.ylabel("train cross-entropy"); plt.legend(); plt.title("Adam vs SGD (equal lr, equal steps)")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

In [ ]:
# Headline-эффект статьи, воспроизведённый контролируемой абляцией.
assert adam_final < floor_ce and sgd_final < floor_ce, "оба оптимизатора обязаны пробить тривиальный пол"
assert adam_final < sgd_final, f"Adam должен дойти до меньшего train loss, чем SGD (adam={adam_final:.4f} sgd={sgd_final:.4f})"
assert adam_speed is not None and (sgd_speed is None or adam_speed < sgd_speed), \
    "Adam должен достигать целевого лосса не медленнее SGD"
rel = (sgd_final - adam_final) / sgd_final
assert rel > 0.20, f"ожидался заметный разрыв, получил rel={rel:.3f}"
assert np.all(np.isfinite(h_adam)) and np.all(np.isfinite(h_sgd)), "non-finite loss"
print(f"ok  Adam < SGD ({adam_final:.4f} < {sgd_final:.4f}), relative improvement {rel*100:.1f}%, faster to target ({adam_speed} vs {sgd_speed})")

## Part 7 — фишка руками: bias correction

Здесь щупаем ровно то, что отличает Adam от «RMSProp с моментом»: коррекцию смещения. Второй контроль — тот же Adam, но `bias_correction=False`. Теория предсказывает: без коррекции первый шаг раздут в $\approx 3.16$ раза (ill-scaled старт), с коррекцией — ограничен $\approx\alpha$. Значит первые эпохи двух прогонов обязаны заметно разойтись, а корректный Adam — не перелетать выше пола.

In [ ]:
p_nobc, h_nobc = train(lambda P: Adam(P, lr=LR, bias_correction=False),
                       Xtr, ytr, epochs=EPOCHS, batch=BATCH, seed=SEED)
early_gap = abs(h_adam[0] - h_nobc[0])
print(f"epoch-1 train CE: corrected={h_adam[0]:.4f}  uncorrected={h_nobc[0]:.4f}  gap={early_gap:.4f}")
print(f"floor CE = {floor_ce:.4f}  (corrected epoch-1 должен быть ниже: без перелёта)")

In [ ]:
# Bias correction меняет раннюю динамику измеримо...
assert early_gap > 0.05, f"bias correction обязана менять ранние шаги (corr ep1={h_adam[0]:.4f} vs uncorr ep1={h_nobc[0]:.4f})"
# ...и корректный первый шаг ограничен: не перелетает выше пола (bounded near lr, no overshoot).
assert h_adam[0] < floor_ce + 1e-6, "корректный первый шаг Adam должен быть ограничен (без overshoot над полом)"
print(f"ok  bias correction меняет ранние шаги (corr ep1={h_adam[0]:.4f}, uncorr ep1={h_nobc[0]:.4f}, gap={early_gap:.4f}); первый шаг ограничен")

## Рефлексия

- **Что забрать в работу.** Adam как дефолт держится не на «магии», а на двух дешёвых per-coordinate статистиках + коррекции старта; когда он всё же ведёт себя странно (взрыв на первых шагах, плато), теперь понятно, за какие ручки дёргать ($\beta_2$, $\epsilon$, warmup вместо/вместе с bias correction).
- **Когда именно тянуться за этим методом.** Плохо-обусловленный, разно-масштабный ландшафт градиентов, где ручной per-layer lr дорог: разреженные фичи, разнородные слои, шумные стохастические градиенты. Там, где ландшафт ровный и хорошо обусловленный, честно настроенный SGD(+momentum) часто обобщает лучше — адаптивность не бесплатна.
- **Что осталось непонятным (следующий узел).** Почему Adam иногда обобщает хуже SGD при равном train loss (generalization gap)? Это ведёт к AdamW (decoupled weight decay) и к amsgrad-поправке сходимости — естественный следующий узел дерева.

---

### ❓ Inline-вопросы (собери ответы здесь)
- **Q0** (AdaGrad vs Adam, сумма vs EMA): ...
- **Q1** (softmax+CE градиент по логитам): ...
- **Q2** ($\epsilon$ в знаменателе, положение): ...

### Going deeper (без баллов)
1. Добавь SGD+momentum как третью кривую — где он между SGD и Adam?
2. Свипни $\beta_2 \in \{0.9, 0.99, 0.999\}$ — как меняется ранняя устойчивость и финальный лосс?
3. Реализуй AdamW (decoupled weight decay) и сравни test accuracy при равном train loss.
4. Замени bias correction на linear warmup первых $k$ шагов — воспроизводит ли эффект?

_Лог экспериментов:_ ...

## SPOILERS — полные тела дыр

Открывай, только если застряла и уже пробовала сама. NaN-дебаг: если лосс улетает в inf, первым делом печатай `self.m`, `self.v` и `np.sqrt(v_hat)` на первых шагах.

<details><summary>trivial_floor</summary>

```python
def trivial_floor(ytr, yte):
    n_classes = int(ytr.max()) + 1
    counts = np.bincount(ytr, minlength=n_classes)
    majority = counts.argmax()
    prior = counts / counts.sum()
    ce = -np.sum(prior * np.log(prior + 1e-12))
    acc = float((yte == majority).mean())
    return ce, acc
```
</details>

<details><summary>backward</summary>

```python
def backward(params, cache, y):
    X, z1, a1, p = cache
    n = y.shape[0]
    dz2 = p.copy()
    dz2[np.arange(n), y] -= 1.0
    dz2 /= n
    grads = {}
    grads["W2"] = a1.T @ dz2
    grads["b2"] = dz2.sum(axis=0)
    da1 = dz2 @ params["W2"].T
    dz1 = da1 * (z1 > 0.0)
    grads["W1"] = X.T @ dz1
    grads["b1"] = dz1.sum(axis=0)
    return grads
```
</details>

<details><summary>train</summary>

```python
def train(make_opt, Xtr, ytr, epochs=30, batch=64, seed=SEED):
    params = init_params(Xtr.shape[1], 32, 10, seed=seed)
    opt = make_opt(params)
    rng = np.random.default_rng(seed)
    n = Xtr.shape[0]
    history = []
    for _ in range(epochs):
        perm = rng.permutation(n)
        for s in range(0, n, batch):
            idx = perm[s:s + batch]
            p, cache = forward(params, Xtr[idx])
            grads = backward(params, cache, ytr[idx])
            opt.step(params, grads)
        p_full, _ = forward(params, Xtr)
        history.append(cross_entropy(p_full, ytr))
    return params, history
```
</details>

<details><summary>Adam.step</summary>

```python
    def step(self, params, grads):
        self.t += 1
        for k in params:
            g = grads[k]
            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g
            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * (g * g)
            if self.bias_correction:
                m_hat = self.m[k] / (1 - self.b1 ** self.t)
                v_hat = self.v[k] / (1 - self.b2 ** self.t)
            else:
                m_hat, v_hat = self.m[k], self.v[k]
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
```
</details>

<details><summary>SGD.step</summary>

```python
    def step(self, params, grads):
        for k in params:
            params[k] -= self.lr * grads[k]
```
</details>